In [ ]:
pip install transformers datasets torch pandas scikit-learn accelerate optuna wandb matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, set_seed, Trainer, 
TrainingArguments, pipeline, EarlyStoppingCallback)
from pathlib import Path
from datasets import Dataset
from sklearn.metrics import average_precision_score, f1_score, classification_report
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import optuna
import wandb
import torch
from optuna.storages import RDBStorage


# Dataset Preperation

In [2]:
from pathlib import Path

main_dir = Path.cwd()

In [3]:
PATH_GAMETOX_TRAIN = "Existing_Datasets/GameTox/train_augmented.csv"
PATH_GAMETOX_VAL = "Existing_Datasets/GameTox/val.csv"

In [4]:
df = pd.read_csv( main_dir / PATH_GAMETOX_TRAIN )
df_2 = pd.read_csv( main_dir / PATH_GAMETOX_VAL )

In [5]:
df["label"] = df["label"].astype(int)
df_2["label"] = df_2["label"].astype(int)

df = df[["message", "label"]]
df_2 = df_2[["message", "label"]]

In [6]:
df = df[df['label'] < 4]
df_2 = df_2[df_2['label'] < 4]

In [7]:
set_seed(42)

dataset_train = Dataset.from_pandas(df)
dataset_val = Dataset.from_pandas(df_2)

In [8]:
PATH_GAMETOX_TEST_TEXT = "Existing_Datasets/GameTox/test_index_text.csv"
PATH_GAMETOX_TEST_LABELS = "Existing_Datasets/GameTox/test_index_label.csv"

In [9]:
df_test_text = pd.read_csv(main_dir / PATH_GAMETOX_TEST_TEXT)
df_test_labels = pd.read_csv(main_dir / PATH_GAMETOX_TEST_LABELS)

In [10]:
df_test = pd.concat([df_test_text['message'], df_test_labels['label']], axis=1)
df_test = df_test[df_test['label'] < 4]
df_test['label'] = df_test['label'].astype(int)

# HateBERT

In [43]:
MODEL_NAME_HATEBERT = "GroNLP/hateBERT"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_HATEBERT)

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_HATEBERT,
    num_labels=4,
    ignore_mismatched_sizes=True
    )

In [12]:
def tokenize(batch):
    return tokenizer(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [14]:
tokenized_dataset_train = dataset_train.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [15]:
tokenized_dataset_val = dataset_val.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/5357 [00:00<?, ? examples/s]

# Optuna Storage for Hyperparameter Search

In [11]:
storage = RDBStorage("sqlite:///optuna_trials.db")

study = optuna.create_study(
    study_name="HateBERT_optuna_study",
    direction="maximize",
    storage=storage,
    load_if_exists=True
)

[I 2026-08-14 10:32:19,121] Using an existing study with name 'HateBERT_optuna_study' instead of creating a new one.


In [12]:
def softmax(x, axis=1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [18]:
def compute_objective(metrics):
    return metrics["eval_macro_F1"]

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    probabilities = softmax(
        logits, 
        axis=1
    )

    n_classes = logits.shape[1]

    ap_scores = []
    for i in range(n_classes):
        y_true = (labels == i).astype(int)
        y_score = probabilities[:, i]
        ap = average_precision_score(y_true, y_score)
        ap_scores.append(ap)

    return {
        "macro_AUPRC": np.mean(ap_scores),

        "macro_F1":
            f1_score(
                labels,
                predictions,
                average="macro"
            )
    }

In [ ]:
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/dja1/.netrc
wandb: Currently logged in as: dj125101308 (dj125101308-university-college-cork) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
wandb.init(project="hf-optuna", name="HateBERT_optuna_study")

training_args = TrainingArguments(
    output_dir="./HateBERT-GameTox-HPS",
    logging_dir="./HateBERT-GameTox-logs",

    num_train_epochs=3,
    logging_strategy="steps",

    eval_strategy="steps",

    save_strategy="steps",

    report_to="wandb",
    run_name="HateBERT_optuna_study"
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/dja1/.netrc.


wandb: Currently logged in as: dj125101308 (dj125101308-university-college-cork) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [42]:
trainer = Trainer(
    model_init=model_init,

    args=training_args,

    train_dataset=
        tokenized_dataset_train,

    eval_dataset=
        tokenized_dataset_val,

    compute_metrics=
        compute_metrics,

    processing_class=
        tokenizer
)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [44]:
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [16, 32, 64]
        ),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
    }

best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=10,
    compute_objective=compute_objective,
    study_name="HateBERT_optuna_study",
    storage="sqlite:///optuna_trials.db",
    load_if_exists=True
)

[I 2026-08-13 13:05:19,762] Using an existing study with name 'HateBERT_optuna_study' instead of creating a new one.


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Macro Auprc,Macro F1
500,0.372100,0.334394,0.590858,0.523074
1000,0.319592,0.330707,0.614119,0.606187
1500,0.285430,0.329509,0.639771,0.629019
2000,0.246598,0.332901,0.644449,0.628510
2500,0.238062,0.312995,0.660476,0.662737
3000,0.196831,0.349510,0.647759,0.652943
3500,0.170740,0.356218,0.645771,0.636166
4000,0.167395,0.361935,0.649888,0.653497
4020,0.167395,0.361845,0.649659,0.654097


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-08-13 15:27:18,466] Trial 21 finished with value: 0.6549227616565174 and parameters: {'learning_rate': 2.9306554505064665e-05, 'per_device_train_batch_size': 64, 'weight_decay': 0.16133574100723583}. Best is trial 21 with value: 0.6549227616565174.


In [17]:
print(study.best_params)

{'learning_rate': 2.9306554505064665e-05, 'per_device_train_batch_size': 64, 'weight_decay': 0.16133574100723583}


# Final Training with Best Hyperparameters

In [13]:
def compute_metrics_final(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    probabilities = softmax(
        logits, 
        axis=1
    )

    n_classes = logits.shape[1]

    ap_scores = []
    for i in range(n_classes):
        y_true = (labels == i).astype(int)
        y_score = probabilities[:, i]
        ap = average_precision_score(y_true, y_score)
        ap_scores.append(ap)

    report = classification_report(
        labels, 
        predictions, 
        output_dict=True, 
        zero_division=0,
        digits=4
    )
    
    per_label_precision = []
    per_label_recall = []
    per_label_f1 = []
    
    for i in range(n_classes):
        per_label_precision.append(report[str(i)]['precision'])
        per_label_recall.append(report[str(i)]['recall'])
        per_label_f1.append(report[str(i)]['f1-score'])

    return {
        "macro_AUPRC": np.mean(ap_scores),

        "macro_F1":
            f1_score(
                labels,
                predictions,
                average="macro"
            ),

        "per_label_precision": per_label_precision,
        "per_label_recall": per_label_recall,
        "per_label_f1": per_label_f1
    }

In [ ]:
training_args_final = TrainingArguments(
    output_dir="./HateBERT-GameTox",

    learning_rate=study.best_params["learning_rate"],

    per_device_train_batch_size=study.best_params["per_device_train_batch_size"],

    weight_decay=study.best_params["weight_decay"],

    metric_for_best_model="macro_F1",
    greater_is_better=True, 

    eval_strategy="steps",

    save_strategy="steps",

    num_train_epochs=3,

    save_total_limit=1
)

In [21]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_HATEBERT)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_HATEBERT,
    num_labels=4,
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
trainer_final = Trainer(
    model=model,

    args=training_args_final,

    train_dataset=
        tokenized_dataset_train,

    eval_dataset=
        tokenized_dataset_val,

    compute_metrics=
        compute_metrics_final,

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)


trainer_final.train()

Step,Training Loss,Validation Loss,Macro Auprc,Macro F1,Per Label Precision,Per Label Recall,Per Label F1
500,0.186455,0.359418,0.618725,0.614350,"[0.9417409184372858, 0.6961869618696187, 0.5584415584415584, 0.6153846153846154]","[0.9478040928949184, 0.7648648648648648, 0.36752136752136755, 0.23529411764705882]","[0.9447627779051112, 0.7289117836445589, 0.44329896907216493, 0.3404255319148936]"
1000,0.235462,0.341106,0.638835,0.646768,"[0.9497917630726516, 0.7128834355828221, 0.5347593582887701, 0.42424242424242425]","[0.9438951483099564, 0.7851351351351351, 0.42735042735042733, 0.4117647058823529]","[0.9468342751701072, 0.7472668810289389, 0.4750593824228028, 0.417910447761194]"
1500,0.188561,0.364246,0.631269,0.638015,"[0.9472836095764272, 0.7218628719275549, 0.49765258215962443, 0.4444444444444444]","[0.946194527477581, 0.754054054054054, 0.452991452991453, 0.35294117647058826]","[0.9467387553203727, 0.7376074025115664, 0.4742729306487696, 0.39344262295081966]"
2000,0.159187,0.362554,0.633328,0.647970,"[0.9473684210526315, 0.720253164556962, 0.5133689839572193, 0.4827586206896552]","[0.9478040928949184, 0.768918918918919, 0.41025641025641024, 0.4117647058823529]","[0.9475862068965517, 0.7437908496732026, 0.45605700712589076, 0.4444444444444444]"
2010,0.159187,0.362575,0.633337,0.647970,"[0.9473684210526315, 0.720253164556962, 0.5133689839572193, 0.4827586206896552]","[0.9478040928949184, 0.768918918918919, 0.41025641025641024, 0.4117647058823529]","[0.9475862068965517, 0.7437908496732026, 0.45605700712589076, 0.4444444444444444]"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2010, training_loss=0.1922353579037225, metrics={'train_runtime': 2708.6937, 'train_samples_per_second': 47.486, 'train_steps_per_second': 0.742, 'total_flos': 3.3843267214848e+16, 'train_loss': 0.1922353579037225, 'epoch': 3.0})

In [40]:
print(trainer_final.state.best_model_checkpoint)

./HateBERT-GameTox/checkpoint-2000


In [44]:
model = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "HateBERT-GameTox" / "checkpoint-2000"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_HATEBERT)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [45]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

In [51]:
preds = classifier(df_test['message'].tolist())
preds = [(model.config.label2id[pred['label']]) for pred in preds]

In [52]:
print(classification_report(df_test['label'], preds, digits=4, zero_division=True))

              precision    recall  f1-score   support

           0     0.9494    0.9536    0.9515      4351
           1     0.7569    0.7803    0.7684       742
           2     0.4752    0.4085    0.4394       235
           3     0.5556    0.4167    0.4762        36

    accuracy                         0.9021      5364
   macro avg     0.6843    0.6398    0.6589      5364
weighted avg     0.8994    0.9021    0.9005      5364



# Detoxify

In [15]:
MODEL_NAME_DETOXIFY = "unitary/toxic-bert"

In [16]:
tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_NAME_DETOXIFY)

model_2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DETOXIFY,
    num_labels=4,
    ignore_mismatched_sizes=True,
    problem_type="single_label_classification"
)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `6`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [17]:
def tokenize_2(batch):
    return tokenizer_2(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [18]:
tokenized_dataset_train = dataset_train.map(
    tokenize_2,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [19]:
tokenized_dataset_val = dataset_train.map(
    tokenize_2,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [20]:
training_args_2 = TrainingArguments(
    output_dir="./Detoxify-GameTox",

    learning_rate=study.best_params["learning_rate"],

    per_device_train_batch_size=study.best_params["per_device_train_batch_size"],

    weight_decay=study.best_params["weight_decay"],

    metric_for_best_model="macro_F1",
    greater_is_better=True, 

    eval_strategy="steps",

    save_strategy="steps",

    num_train_epochs=3,

    save_total_limit=1
)

In [21]:
trainer_2 = Trainer(
    model=model_2,

    args=training_args_2,

    train_dataset=
        tokenized_dataset_train,

    eval_dataset=
        tokenized_dataset_val,

    compute_metrics=
        compute_metrics_final,

    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)


trainer_2.train()

[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss,Macro Auprc,Macro F1,Per Label Precision,Per Label Recall,Per Label F1
500,0.351394,0.248371,0.693315,0.669436,"[0.948570466597458, 0.8091723774380601, 0.5662563359884142, 0.6049382716049383]","[0.9715780095985286, 0.7772151898734178, 0.41728922091782283, 0.35125448028673834]","[0.9599363979670065, 0.7928719008264463, 0.4804915514592934, 0.4444444444444444]"
1000,0.261123,0.203933,0.771459,0.728275,"[0.9660135945621752, 0.8156786524133463, 0.6861538461538461, 0.4728682170542636]","[0.972037819352243, 0.849957805907173, 0.4759871931696905, 0.6559139784946236]","[0.9690163441291487, 0.8324654930159517, 0.5620667926906112, 0.5495495495495496]"
1500,0.214721,0.163254,0.832613,0.794327,"[0.97566937965546, 0.8642999336429993, 0.6760797342192691, 0.6777777777777778]","[0.9749403684225652, 0.8793248945147679, 0.651547491995731, 0.6559139784946236]","[0.9753047378104875, 0.8717476784070944, 0.6635869565217392, 0.6666666666666666]"
2000,0.186412,0.144965,0.861059,0.818899,"[0.9782177648783011, 0.8683013965573239, 0.7463325183374083, 0.7127659574468085]","[0.9782739891369946, 0.9024472573839663, 0.651547491995731, 0.7204301075268817]","[0.9782458761997816, 0.8850451046925433, 0.6957264957264957, 0.7165775401069518]"
2010,0.186412,0.144952,0.861056,0.818846,"[0.9782171389160297, 0.8683013965573239, 0.7458766035430666, 0.7127659574468085]","[0.9782452510273875, 0.9024472573839663, 0.651547491995731, 0.7204301075268817]","[0.9782311947697392, 0.8850451046925433, 0.6955283395044147, 0.7165775401069518]"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2010, training_loss=0.2529485153321603, metrics={'train_runtime': 4068.1985, 'train_samples_per_second': 31.617, 'train_steps_per_second': 0.494, 'total_flos': 3.3843267214848e+16, 'train_loss': 0.2529485153321603, 'epoch': 3.0})

In [22]:
print(trainer_2.state.best_model_checkpoint)

./Detoxify-GameTox/checkpoint-2000


In [20]:
model_2 = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "Detoxify-GameTox" / "checkpoint-2000"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [21]:
classifier = pipeline("text-classification", model=model_2, tokenizer=tokenizer_2)

In [22]:
preds = classifier(df_test['message'].tolist())
preds = [(model_2.config.label2id[pred['label']]) for pred in preds]

In [23]:
print(classification_report(df_test['label'], preds, digits=4, zero_division=True))

              precision    recall  f1-score   support

           0     0.9472    0.9556    0.9514      4351
           1     0.7603    0.7695    0.7649       742
           2     0.4847    0.4043    0.4408       235
           3     0.6296    0.4722    0.5397        36

    accuracy                         0.9025      5364
   macro avg     0.7054    0.6504    0.6742      5364
weighted avg     0.8989    0.9025    0.9005      5364



# BERT

In [24]:
MODEL_NAME_BERT = "google-bert/bert-base-uncased"

In [25]:
tokenizer_3 = AutoTokenizer.from_pretrained(MODEL_NAME_BERT)

model_3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_BERT,
    num_labels=4,
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
def tokenize_3(batch):
    return tokenizer_3(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [27]:
tokenized_dataset_train = dataset_train.map(
    tokenize_3,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [28]:
tokenized_dataset_val = dataset_train.map(
    tokenize_3,
    batched=True
)

Map:   0%|          | 0/42875 [00:00<?, ? examples/s]

In [29]:
training_args_3 = TrainingArguments(
    output_dir="./GoogleBERT-GameTox",

    learning_rate=study.best_params["learning_rate"],

    per_device_train_batch_size=study.best_params["per_device_train_batch_size"],

    weight_decay=study.best_params["weight_decay"],

    metric_for_best_model="macro_F1",
    greater_is_better=True, 

    eval_strategy="steps",

    save_strategy="steps",

    num_train_epochs=3,

    save_total_limit=1
)